# Reward Group Analysis

Notebook ini menganalisis **environment reward** untuk eksperimen `1%` pada tiga partition family:

- `Random`
- `Profile-Based Partition`
- `Single Attribute Partition`

User dikelompokkan menjadi:

- `SRU` = Sustained Retain Users
- `RRU` = Regressed Retain Users
- `EFU` = Effectively Forgotten Users

Reward dihitung mengikuti environment training:

- rating `5` -> `1.0`
- rating `4` -> `0.8`
- rating `< 4` atau tidak ada di future -> `0.0`


In [ ]:
from pathlib import Path
import hashlib
import math
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'figure.figsize': (10.5, 6.0),
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.edgecolor': '#D0D7DE',
    'axes.grid': True,
    'grid.color': '#E5E7EB',
    'grid.linewidth': 0.8,
    'grid.alpha': 1.0,
    'axes.axisbelow': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 15,
    'axes.labelsize': 11,
    'legend.frameon': False,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Trebuchet MS', 'Verdana', 'DejaVu Sans']
})

SEED = 97620260313
K_TARGET = 10
FORGET_PERCENTAGE = 1
ANALYSIS_THRESHOLD_PP = 5.0
TARGET_METHODS = ['Ye_multi', 'New_True_inf', 'Gradient_Ascent']
METHOD_LABELS = {
    'Ye_multi': 'Metode Terdahulu',
    'New_True_inf': 'Metode Diusulkan',
    'Gradient_Ascent': 'Baseline',
}
METHOD_COLORS = {
    'Gradient_Ascent': '#7C3AED',
    'Ye_multi': '#EAB308',
    'New_True_inf': '#2563EB',
}
PARTITION_COLORS = {
    'Random': '#2563EB',
    'Profile-Based Partition': '#DC2626',
    'Single Attribute Partition': '#059669',
}
GROUP_COLORS = {
    'SRU': '#0F766E',
    'RRU': '#D97706',
    'EFU': '#DC2626',
}
AGE_LABELS = {
    1: 'Under 18',
    18: '18-24',
    25: '25-34',
    35: '35-44',
    45: '45-49',
    50: '50-55',
    56: '56+',
}
OCCUPATION_LABELS = {
    0: 'Other / not specified',
    1: 'Academic/Educator',
    2: 'Artist',
    3: 'Clerical/Admin',
    4: 'College/Grad Student',
    5: 'Customer Service',
    6: 'Doctor/Health Care',
    7: 'Executive/Managerial',
    8: 'Farmer',
    9: 'Homemaker',
    10: 'K-12 Student',
    11: 'Lawyer',
    12: 'Programmer',
    13: 'Retired',
    14: 'Sales/Marketing',
    15: 'Scientist',
    16: 'Self-Employed',
    17: 'Technician/Engineer',
    18: 'Tradesman/Craftsman',
    19: 'Unemployed',
    20: 'Writer',
}

REPO_ROOT = Path(r'C:/Bob')
DATA_DIR = Path(r'C:/Bob/ml-1m')
UGP_METRICS_PATH = Path(r'D:/Bob_Skripsi_Do Not Delete/results_ugp_analysis/metrics/relearn_metrics.csv')
RANDOM_RESULTS_PATH = Path(r'C:/Bob/results/1_percent/tuning_full_results.csv')
PROFILE_RESULTS_PATH = Path(r'D:/Bob_Skripsi_Do Not Delete/results_demography/1_percent/tuning_full_results.csv')
OUTPUT_ROOT = Path(r'D:/Bob_Skripsi_Do Not Delete/results_ugp_analysis/UGP_Post_Analysis/Reward_Analysis')
FIG_DIR = OUTPUT_ROOT / 'figures'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def make_seed(*args):
    key = f"{SEED}|{'|'.join(str(a) for a in args)}".encode('utf-8')
    return int(hashlib.sha256(key).hexdigest()[:8], 16)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed % (2**31 - 1))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed % (2**31 - 1))

set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

print(f'REPO_ROOT            : {REPO_ROOT}')
print(f'DATA_DIR             : {DATA_DIR}')
print(f'UGP_METRICS_PATH     : {UGP_METRICS_PATH}')
print(f'RANDOM_RESULTS_PATH  : {RANDOM_RESULTS_PATH}')
print(f'PROFILE_RESULTS_PATH : {PROFILE_RESULTS_PATH}')
print(f'OUTPUT_ROOT          : {OUTPUT_ROOT}')
print(f'DEVICE               : {DEVICE}')


In [ ]:
def compute_drop_pp(base_col, after_col, frame):
    return (frame[base_col] - frame[after_col]) * 100.0

def format_setting_label(setting_type, raw_value, setting_label=None):
    if setting_type == 'gender':
        return 'Female' if str(raw_value) == 'F' else 'Male'
    if setting_type == 'age':
        return AGE_LABELS.get(int(float(raw_value)), str(raw_value))
    if setting_type == 'occupation':
        occ = int(float(raw_value))
        return OCCUPATION_LABELS.get(occ, f'Occupation {occ}')
    if setting_label:
        return str(setting_label).replace('_', ' ').title()
    return str(raw_value)

def load_partition_results(path, family_label):
    df = pd.read_csv(path).copy()
    df = df[df['method'].isin(TARGET_METHODS)].copy()
    df = df[df['K'] == K_TARGET].copy()
    numeric_cols = [
        'train_lr', 'gamma', 'hidden_dim', 'train_batch', 'unlearn_lr', 'unlearn_iters', 'lambda_retain',
        'retain_Hit', 'forget_Hit', 'base_retain_Hit', 'base_forget_Hit', 'retain_NDCG', 'forget_NDCG',
        'base_retain_NDCG', 'base_forget_NDCG'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['retain_drop_hit_pp'] = compute_drop_pp('base_retain_Hit', 'retain_Hit', df)
    df['forget_drop_hit_pp'] = compute_drop_pp('base_forget_Hit', 'forget_Hit', df)
    df['partition_family'] = family_label
    df['setting_type'] = family_label.lower().replace(' ', '_')
    df['setting_value_raw'] = FORGET_PERCENTAGE
    df['setting_label'] = f'{family_label} 1%'
    df['setting_display'] = f'{family_label} 1%'
    return df

def pick_best_partition_rows(df, threshold):
    constrained = df[df['retain_drop_hit_pp'] <= threshold].copy()
    if constrained.empty:
        return pd.DataFrame(columns=list(df.columns) + ['analysis_threshold_pp'])
    constrained = constrained.sort_values(
        ['method', 'forget_drop_hit_pp', 'lambda_retain'],
        ascending=[True, False, True],
        kind='mergesort'
    )
    picked = constrained.groupby('method', as_index=False).first()
    picked['analysis_threshold_pp'] = float(threshold)
    return picked

def load_ugp_metrics(path):
    df = pd.read_csv(path).copy()
    df = df[df['method'].isin(TARGET_METHODS)].copy()
    df = df[df['K'] == K_TARGET].copy()
    numeric_cols = [
        'forget_user_count', 'retain_user_count', 'retain_Hit', 'forget_Hit', 'base_retain_Hit', 'base_forget_Hit',
        'retain_NDCG', 'forget_NDCG', 'base_retain_NDCG', 'base_forget_NDCG', 'lambda_retain', 'hidden_dim',
        'train_batch', 'train_lr', 'gamma', 'unlearn_lr', 'unlearn_iters', 'base_threshold_pp'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['retain_drop_hit_pp'] = compute_drop_pp('base_retain_Hit', 'retain_Hit', df)
    df['forget_drop_hit_pp'] = compute_drop_pp('base_forget_Hit', 'forget_Hit', df)
    df['partition_family'] = 'Single Attribute Partition'
    df['setting_display'] = [format_setting_label(st, rv, sl) for st, rv, sl in zip(df['setting_type'], df['setting_value_raw'], df['setting_label'])]
    return df

def load_data(data_dir):
    ratings_df = pd.read_csv(data_dir / 'ratings.dat', sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp'])
    movies_df = pd.read_csv(data_dir / 'movies.dat', sep='::', engine='python', names=['movie_id', 'title', 'genres'], encoding='ISO-8859-1')
    users_df = pd.read_csv(data_dir / 'users.dat', sep='::', engine='python', names=['user_id', 'gender', 'age', 'occupation', 'zip'])
    return ratings_df, movies_df, users_df

class PolicyNet(nn.Module):
    def __init__(self, state_dim, num_actions, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.logits = nn.Linear(hidden_dim, num_actions)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.logits(x)

def prepare_environment():
    ratings_df, movies_df, users_df = load_data(DATA_DIR)
    sample_users = ratings_df['user_id'].unique().tolist()
    pilot_ratings_all = ratings_df[ratings_df['user_id'].isin(sample_users)].copy()
    pilot_ratings_all.sort_values(['user_id', 'timestamp'], inplace=True)
    pilot_users_df = users_df[users_df['user_id'].isin(sample_users)]
    user_cats = pilot_users_df[['user_id', 'gender', 'age', 'occupation']].copy()
    oh = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    user_cat_mat = oh.fit_transform(user_cats[['gender', 'age', 'occupation']])
    user_feat_df = pd.DataFrame(user_cat_mat, index=user_cats['user_id'])
    all_genres = sorted({g for s in movies_df['genres'].astype(str) for g in s.split('|')})
    genre_to_idx = {g: i for i, g in enumerate(all_genres)}
    num_genres = len(all_genres)

    def movie_genre_vector(genres_str):
        v = np.zeros(num_genres, dtype=np.float32)
        for g in str(genres_str).split('|'):
            if g in genre_to_idx:
                v[genre_to_idx[g]] = 1.0
        return v

    movies_df = movies_df.copy()
    movies_df['genre_vec'] = movies_df['genres'].apply(movie_genre_vector)
    movie_genre_map = {
        mid: movies_df.loc[movies_df['movie_id'] == mid, 'genre_vec'].values[0]
        for mid in movies_df['movie_id'].unique()
    }
    state_dim = user_feat_df.shape[1] + num_genres

    def build_state_fn(user_id, watched_movies):
        user_feat = user_feat_df.loc[user_id].values.astype(np.float32)
        pref_vec = np.zeros(num_genres, dtype=np.float32)
        for mid in watched_movies:
            if mid in movie_genre_map:
                pref_vec += movie_genre_map[mid]
        s = pref_vec.sum()
        if s > 0:
            pref_vec /= s
        return np.concatenate([user_feat, pref_vec]).astype(np.float32)

    trajectories_all = [
        {'user_id': uid, 'movies': g['movie_id'].tolist(), 'ratings': g['rating'].tolist()}
        for uid, g in pilot_ratings_all.groupby('user_id') if len(g) >= 5
    ]
    candidate_movies = np.array(sorted(pilot_ratings_all['movie_id'].unique()))
    return {
        'ratings_df': ratings_df,
        'movies_df': movies_df,
        'users_df': users_df,
        'sample_users': sample_users,
        'trajectories_all': trajectories_all,
        'trajectory_by_user': {traj['user_id']: traj for traj in trajectories_all},
        'candidate_movies': candidate_movies,
        'num_actions': len(candidate_movies),
        'state_dim': state_dim,
        'build_state_fn': build_state_fn,
    }

def evaluate_per_user_midpoint(policy_net, trajectories, candidate_movies, build_state_fn, K=10):
    rows = []
    candidate_movies = np.array(candidate_movies)
    policy_net.eval()
    for traj in trajectories:
        uid = traj['user_id']
        movies = traj['movies']
        if len(movies) < 5:
            continue
        mid = len(movies) // 2
        state = build_state_fn(uid, movies[:mid])
        future = set(movies[mid:])
        with torch.no_grad():
            logits = policy_net(torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)).squeeze(0)
            topk_idx = torch.topk(logits, K).indices.cpu().numpy()
        topk_movies = set(candidate_movies[topk_idx])
        rows.append({'uid': uid, 'hit': len(topk_movies.intersection(future)) > 0})
    return pd.DataFrame(rows)

def load_policy(model_path, state_dim, num_actions, hidden_dim):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Model path tidak ditemukan: {model_path}')
    net = PolicyNet(state_dim, num_actions, hidden_dim=int(hidden_dim)).to(DEVICE)
    net.load_state_dict(torch.load(model_path, map_location=DEVICE))
    net.eval()
    return net

def build_random_split(sample_users):
    users = np.array(list(sample_users), dtype=int)
    set_seed(SEED)
    np.random.shuffle(users)
    split_amt = int(np.round(FORGET_PERCENTAGE / 100 * len(users)))
    return users[:split_amt], users[split_amt:]

def get_multiplier_demo(gender, age, occupation):
    mult = 1.0
    if gender == 'F':
        mult *= 2.0
    if age == 1:
        mult *= 1.8
    elif age == 56:
        mult *= 1.5
    elif age == 50:
        mult *= 1.3
    elif age == 18:
        mult *= 0.8
    if occupation in [0, 1, 10, 19]:
        mult *= 1.5
    elif occupation in [2, 14, 20]:
        mult *= 1.3
    elif occupation in [3, 4, 5]:
        mult *= 1.0
    return mult

def build_profile_split(users_df, sample_users):
    users_sorted = np.array(sorted(sample_users), dtype=int)
    users_meta = users_df[users_df['user_id'].isin(sample_users)].set_index('user_id')
    user_mult = []
    for uid in users_sorted:
        row = users_meta.loc[uid]
        user_mult.append((uid, get_multiplier_demo(row['gender'], row['age'], row['occupation'])))
    mean_mult = float(np.mean([m for _, m in user_mult]))
    base_prob = (FORGET_PERCENTAGE / 100) / mean_mult
    probs = np.array([min(base_prob * m, 1.0) for _, m in user_mult], dtype=np.float64)
    probs /= probs.sum()
    split_amt = int(np.round(FORGET_PERCENTAGE / 100 * len(users_sorted)))
    rng = np.random.default_rng(make_seed('profile_split', FORGET_PERCENTAGE))
    forget_users = np.sort(rng.choice(users_sorted, size=split_amt, replace=False, p=probs))
    retain_users = np.array(sorted(set(users_sorted.tolist()) - set(forget_users.tolist())), dtype=int)
    return forget_users, retain_users

def build_single_attribute_split(users_df, sample_users, setting_type, setting_value_raw, target_count=60):
    users_meta = users_df[users_df['user_id'].isin(sample_users)].copy().set_index('user_id').sort_index()
    if setting_type == 'gender':
        subset = users_meta[users_meta['gender'] == str(setting_value_raw)]
    elif setting_type == 'age':
        subset = users_meta[users_meta['age'] == int(float(setting_value_raw))]
    elif setting_type == 'occupation':
        subset = users_meta[users_meta['occupation'] == int(float(setting_value_raw))]
    else:
        raise ValueError(f'Unsupported setting_type: {setting_type}')
    forget_users = subset.index.to_numpy(dtype=int)[:min(target_count, len(subset))]
    retain_users = np.array(sorted(set(sample_users) - set(forget_users.tolist())), dtype=int)
    return forget_users, retain_users

def rating_to_reward(rating):
    if rating >= 5:
        return 1.0
    if rating >= 4:
        return float(rating) / 5.0
    return 0.0

def simulate_episode_reward_for_traj(policy_net, traj, candidate_movies, build_state_fn):
    uid = traj['user_id']
    movies = traj['movies']
    ratings = traj['ratings']
    t = 1
    future_dict = {m: r for m, r in zip(movies[t:], ratings[t:])}
    state = build_state_fn(uid, movies[:t])
    total_reward = 0.0
    policy_net.eval()
    while True:
        with torch.no_grad():
            logits = policy_net(torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)).squeeze(0)
            action_idx = int(torch.argmax(logits).item())
        rec_movie = candidate_movies[action_idx]
        if rec_movie in future_dict:
            total_reward += rating_to_reward(future_dict[rec_movie])
        t += 1
        done = t >= len(movies)
        if done:
            break
        future_dict = {m: r for m, r in zip(movies[t:], ratings[t:])}
        state = build_state_fn(uid, movies[:t])
    return total_reward

MODEL_ANALYSIS_CACHE = {}

def get_model_user_analysis(model_path, hidden_dim, trajectories_all, candidate_movies, build_state_fn, state_dim, num_actions):
    cache_key = (str(model_path), int(hidden_dim))
    if cache_key in MODEL_ANALYSIS_CACHE:
        return MODEL_ANALYSIS_CACHE[cache_key]
    net = load_policy(str(model_path), state_dim, num_actions, hidden_dim)
    midpoint_df = evaluate_per_user_midpoint(net, trajectories_all, candidate_movies, build_state_fn, K=K_TARGET)
    reward_rows = []
    for traj in tqdm(trajectories_all, desc=f'Reward replay: {Path(str(model_path)).name}', leave=False):
        reward_rows.append({
            'uid': traj['user_id'],
            'episode_reward': simulate_episode_reward_for_traj(net, traj, candidate_movies, build_state_fn),
        })
    reward_df = pd.DataFrame(reward_rows)
    MODEL_ANALYSIS_CACHE[cache_key] = (midpoint_df, reward_df)
    del net
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return MODEL_ANALYSIS_CACHE[cache_key]

def summarize_group(case_meta, user_group, uids, reward_before_df, reward_after_df):
    if len(uids) == 0:
        return {
            **case_meta,
            'user_group': user_group,
            'user_count': 0,
            'mean_reward_before': np.nan,
            'mean_reward_after': np.nan,
            'mean_reward_delta': np.nan,
        }
    reward_df = reward_before_df.merge(reward_after_df, on='uid', suffixes=('_before', '_after'))
    reward_df = reward_df[reward_df['uid'].isin(uids)].copy()
    reward_df['reward_delta'] = reward_df['episode_reward_after'] - reward_df['episode_reward_before']
    return {
        **case_meta,
        'user_group': user_group,
        'user_count': int(len(reward_df)),
        'mean_reward_before': float(reward_df['episode_reward_before'].mean()),
        'mean_reward_after': float(reward_df['episode_reward_after'].mean()),
        'mean_reward_delta': float(reward_df['reward_delta'].mean()),
    }

def safe_weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not mask.any():
        return np.nan
    return float(np.average(values[mask], weights=weights[mask]))

def add_bar_labels(ax, bars, fmt='{:.2f}', suffix=''):
    for bar in bars:
        height = float(bar.get_height())
        if not np.isfinite(height):
            continue
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + max(0.02, abs(height) * 0.03),
            f'{fmt.format(height)}{suffix}',
            ha='center', va='bottom', fontsize=9, color='#334155'
        )


In [ ]:
sim_env = prepare_environment()
ratings_df = sim_env['ratings_df']
movies_df = sim_env['movies_df']
users_df = sim_env['users_df']
sample_users = sim_env['sample_users']
trajectories_all = sim_env['trajectories_all']
trajectory_by_user = sim_env['trajectory_by_user']
candidate_movies = sim_env['candidate_movies']
num_actions = sim_env['num_actions']
state_dim = sim_env['state_dim']
build_state_fn = sim_env['build_state_fn']

random_results_k10 = load_partition_results(RANDOM_RESULTS_PATH, 'Random')
profile_results_k10 = load_partition_results(PROFILE_RESULTS_PATH, 'Profile-Based Partition')
ugp_results_k10 = load_ugp_metrics(UGP_METRICS_PATH)

selected_random_df = pick_best_partition_rows(random_results_k10, ANALYSIS_THRESHOLD_PP)
selected_profile_df = pick_best_partition_rows(profile_results_k10, ANALYSIS_THRESHOLD_PP)
selected_ugp_df = pick_best_partition_rows(ugp_results_k10, ANALYSIS_THRESHOLD_PP)

selected_random_df['source_base_threshold_pp'] = selected_random_df['analysis_threshold_pp']
selected_profile_df['source_base_threshold_pp'] = selected_profile_df['analysis_threshold_pp']
selected_ugp_df['source_base_threshold_pp'] = selected_ugp_df['base_threshold_pp']

selected_random_df['method_label'] = selected_random_df['method'].map(METHOD_LABELS)
selected_profile_df['method_label'] = selected_profile_df['method'].map(METHOD_LABELS)
selected_ugp_df['method_label'] = selected_ugp_df['method'].map(METHOD_LABELS)

random_cases = selected_random_df[[
    'partition_family', 'method', 'method_label', 'analysis_threshold_pp', 'source_base_threshold_pp',
    'setting_type', 'setting_label', 'setting_value_raw', 'setting_display', 'hidden_dim',
    'trained_model_path', 'unlearned_model_path'
]].copy()
profile_cases = selected_profile_df[[
    'partition_family', 'method', 'method_label', 'analysis_threshold_pp', 'source_base_threshold_pp',
    'setting_type', 'setting_label', 'setting_value_raw', 'setting_display', 'hidden_dim',
    'trained_model_path', 'unlearned_model_path'
]].copy()
ugp_cases = selected_ugp_df[[
    'partition_family', 'method', 'method_label', 'analysis_threshold_pp', 'source_base_threshold_pp',
    'setting_type', 'setting_label', 'setting_value_raw', 'setting_display', 'hidden_dim',
    'trained_model_path', 'unlearned_model_path'
]].copy()

selected_cases = pd.concat([random_cases, profile_cases, ugp_cases], ignore_index=True)
selected_cases['case_id'] = [
    f"{pf}|{m}|thr{thr:.1f}|{sl}"
    for pf, m, thr, sl in zip(
        selected_cases['partition_family'],
        selected_cases['method'],
        selected_cases['analysis_threshold_pp'],
        selected_cases['setting_label'],
    )
]
selected_cases = selected_cases.sort_values(['partition_family', 'method', 'setting_label']).reset_index(drop=True)

selected_cases.to_csv(TABLE_DIR / 'selected_reward_cases.csv', index=False)
selection_counts = selected_cases.groupby(['partition_family', 'method']).size().rename('count').reset_index()
print(f'Total selected cases: {len(selected_cases)}')
display(selection_counts)
assert len(selected_cases) == 9, f'Expected 9 selected cases, got {len(selected_cases)}'
assert selection_counts['count'].eq(1).all(), 'Each partition_family x method must have exactly 1 selected case'
display(
    selected_cases[[
        'partition_family', 'method', 'analysis_threshold_pp', 'setting_type', 'setting_label',
        'setting_display', 'forget_drop_hit_pp', 'retain_drop_hit_pp', 'trained_model_path', 'unlearned_model_path'
    ]].sort_values(['partition_family', 'method']).reset_index(drop=True)
)


In [ ]:
case_rows = []

for row in tqdm(selected_cases.to_dict('records'), desc='Analyzing reward groups'):
    family = row['partition_family']
    if family == 'Random':
        forget_users, retain_users = build_random_split(sample_users)
    elif family == 'Profile-Based Partition':
        forget_users, retain_users = build_profile_split(users_df, sample_users)
    elif family == 'Single Attribute Partition':
        forget_users, retain_users = build_single_attribute_split(users_df, sample_users, row['setting_type'], row['setting_value_raw'])
    else:
        raise ValueError(f'Unsupported partition family: {family}')

    forget_user_set = set(np.array(forget_users, dtype=int).tolist())
    retain_user_set = set(np.array(retain_users, dtype=int).tolist())

    base_eval_df, base_reward_df = get_model_user_analysis(
        row['trained_model_path'], row['hidden_dim'], trajectories_all, candidate_movies, build_state_fn, state_dim, num_actions
    )
    after_eval_df, after_reward_df = get_model_user_analysis(
        row['unlearned_model_path'], row['hidden_dim'], trajectories_all, candidate_movies, build_state_fn, state_dim, num_actions
    )

    retain_eval = (
        base_eval_df[base_eval_df['uid'].isin(retain_user_set)]
        .merge(after_eval_df[after_eval_df['uid'].isin(retain_user_set)], on='uid', suffixes=('_before', '_after'))
    )
    forget_eval = (
        base_eval_df[base_eval_df['uid'].isin(forget_user_set)]
        .merge(after_eval_df[after_eval_df['uid'].isin(forget_user_set)], on='uid', suffixes=('_before', '_after'))
    )
    retain_eval['hit_before'] = retain_eval['hit_before'].astype(bool)
    retain_eval['hit_after'] = retain_eval['hit_after'].astype(bool)
    forget_eval['hit_before'] = forget_eval['hit_before'].astype(bool)
    forget_eval['hit_after'] = forget_eval['hit_after'].astype(bool)

    sru_uids = retain_eval[(retain_eval['hit_before']) & (retain_eval['hit_after'])]['uid'].tolist()
    rru_uids = retain_eval[(retain_eval['hit_before']) & (~retain_eval['hit_after'])]['uid'].tolist()
    efu_uids = forget_eval[(forget_eval['hit_before']) & (~forget_eval['hit_after'])]['uid'].tolist()

    case_meta = {
        'case_id': row['case_id'],
        'partition_family': row['partition_family'],
        'method': row['method'],
        'method_label': row['method_label'],
        'analysis_threshold_pp': float(row['analysis_threshold_pp']),
        'source_base_threshold_pp': float(row['source_base_threshold_pp']),
        'setting_type': row['setting_type'],
        'setting_label': row['setting_label'],
        'setting_display': row['setting_display'],
    }

    case_rows.append(summarize_group(case_meta, 'SRU', sru_uids, base_reward_df, after_reward_df))
    case_rows.append(summarize_group(case_meta, 'RRU', rru_uids, base_reward_df, after_reward_df))
    case_rows.append(summarize_group(case_meta, 'EFU', efu_uids, base_reward_df, after_reward_df))

reward_group_case_level = pd.DataFrame(case_rows)
reward_group_case_level.to_csv(TABLE_DIR / 'reward_group_case_level.csv', index=False)

summary_rows = []
for keys, group in reward_group_case_level.groupby(['partition_family', 'method', 'method_label', 'analysis_threshold_pp', 'user_group'], sort=True):
    partition_family, method, method_label, analysis_threshold_pp, user_group = keys
    summary_rows.append({
        'partition_family': partition_family,
        'method': method,
        'method_label': method_label,
        'analysis_threshold_pp': float(analysis_threshold_pp),
        'user_group': user_group,
        'num_cases': int(group['case_id'].nunique()),
        'mean_user_count': float(group['user_count'].mean()),
        'total_user_count': int(group['user_count'].sum()),
        'mean_reward_before': safe_weighted_mean(group['mean_reward_before'], group['user_count']),
        'mean_reward_after': safe_weighted_mean(group['mean_reward_after'], group['user_count']),
        'mean_reward_delta': safe_weighted_mean(group['mean_reward_delta'], group['user_count']),
    })

reward_group_summary = pd.DataFrame(summary_rows)
reward_group_summary.to_csv(TABLE_DIR / 'reward_group_summary.csv', index=False)

reward_group_counts_only = reward_group_summary[[
    'partition_family', 'method', 'method_label', 'analysis_threshold_pp', 'user_group',
    'num_cases', 'mean_user_count', 'total_user_count'
]].copy()
reward_group_counts_only.to_csv(TABLE_DIR / 'reward_group_counts_only.csv', index=False)

display(reward_group_case_level)
display(reward_group_summary)

overall_reward_delta = []
for keys, group in reward_group_case_level.groupby(['partition_family', 'method', 'method_label', 'user_group'], sort=True):
    partition_family, method, method_label, user_group = keys
    overall_reward_delta.append({
        'partition_family': partition_family,
        'method': method,
        'method_label': method_label,
        'user_group': user_group,
        'mean_reward_delta': safe_weighted_mean(group['mean_reward_delta'], group['user_count']),
        'mean_user_count': float(group['user_count'].mean()),
    })
overall_reward_delta = pd.DataFrame(overall_reward_delta)

group_order = ['SRU', 'RRU', 'EFU']
partition_order = ['Random', 'Profile-Based Partition', 'Single Attribute Partition']
method_order = TARGET_METHODS
x = np.arange(len(group_order))
width = 0.24

fig, axes = plt.subplots(1, 3, figsize=(17, 5.8), sharey=True)
for ax, method in zip(axes, method_order):
    subset = overall_reward_delta[overall_reward_delta['method'] == method].copy()
    for idx, family in enumerate(partition_order):
        fam_df = subset[subset['partition_family'] == family].set_index('user_group').reindex(group_order).reset_index()
        bars = ax.bar(x + (idx - 1) * width, fam_df['mean_reward_delta'], width=width, color=PARTITION_COLORS[family], label=family)
        add_bar_labels(ax, bars)
    ax.set_title(METHOD_LABELS[method], loc='left', color=METHOD_COLORS[method])
    ax.set_xticks(x)
    ax.set_xticklabels(group_order)
    ax.set_xlabel('User group')
axes[0].set_ylabel('Mean reward delta')
fig.suptitle('Reward Delta by User Group', x=0.06, ha='left', y=1.03, fontsize=15)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, bbox_to_anchor=(0.5, 1.02), loc='lower center')
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(FIG_DIR / 'reward_delta_by_group_method.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(17, 5.8), sharey=True)
for ax, method in zip(axes, method_order):
    subset = overall_reward_delta[overall_reward_delta['method'] == method].copy()
    for idx, family in enumerate(partition_order):
        fam_df = subset[subset['partition_family'] == family].set_index('user_group').reindex(group_order).reset_index()
        bars = ax.bar(x + (idx - 1) * width, fam_df['mean_user_count'], width=width, color=PARTITION_COLORS[family], label=family)
        add_bar_labels(ax, bars, fmt='{:.1f}')
    ax.set_title(METHOD_LABELS[method], loc='left', color=METHOD_COLORS[method])
    ax.set_xticks(x)
    ax.set_xticklabels(group_order)
    ax.set_xlabel('User group')
axes[0].set_ylabel('Mean user count per case')
fig.suptitle('User Counts by Group', x=0.06, ha='left', y=1.03, fontsize=15)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, bbox_to_anchor=(0.5, 1.02), loc='lower center')
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(FIG_DIR / 'reward_group_counts_by_method.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

print('Reward analysis selesai. Tables dan figures sudah disimpan.')
